# 02 — Data Cleaning
### Landslide Risk Monitoring | SIH 26001 | NER

This notebook demonstrates the **complete data cleaning pipeline** step by step.  
Production cleaning logic lives in `ml/preprocessing/cleaning.py` — this notebook shows it in action.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 4)
print('✓ Imports OK | Repo root:', REPO_ROOT)

## 1. Load Raw Dataset

In [ ]:
from ml.preprocessing.cleaning import load_data, clean_data, COLUMN_RANGES
from ml.datasets.generate_sample import generate_sample_dataset

SAMPLE_PATH = REPO_ROOT / 'ml' / 'datasets' / 'sample' / 'landslide_sample.csv'

if not SAMPLE_PATH.exists():
    generate_sample_dataset(n_samples=1000)

df_raw = load_data(SAMPLE_PATH)
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

## 2. Before Cleaning — Snapshot

In [ ]:
print('=== BEFORE CLEANING ===')
print(f'Rows              : {len(df_raw)}')
print(f'Duplicate rows    : {df_raw.duplicated().sum()}')
print(f'Total nulls       : {df_raw.isnull().sum().sum()}')
print()
print('Missing values per column:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

## 3. Run Cleaning Pipeline

In [ ]:
df_clean = clean_data(df_raw, verbose=True)
print(f'\nCleaned shape: {df_clean.shape}')

## 4. After Cleaning — Comparison

In [ ]:
comparison = pd.DataFrame({
    'Before': [
        len(df_raw),
        df_raw.duplicated().sum(),
        df_raw.isnull().sum().sum(),
        df_raw.shape[1],
    ],
    'After': [
        len(df_clean),
        df_clean.duplicated().sum(),
        df_clean.isnull().sum().sum(),
        df_clean.shape[1],
    ]
}, index=['Row Count', 'Duplicates', 'Missing Values', 'Columns'])

print('=== CLEANING COMPARISON ===')
print(comparison)

## 5. Visualize Before / After — Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [
    (axes[0], df_raw, 'BEFORE Cleaning'),
    (axes[1], df_clean, 'AFTER Cleaning')
]:
    missing = df.isnull().sum()
    colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing]
    bars = ax.bar(missing.index, missing.values, color=colors)
    ax.set_title(f'Missing Values — {title}', fontweight='bold')
    ax.set_ylabel('Missing Count')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, missing.values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Validate Numeric Ranges

In [ ]:
print('=== COLUMN RANGE VALIDATION (after cleaning) ===')
print(f'{"Column":<35} {"Min":>8} {"Max":>8} {"Expected Min":>14} {"Expected Max":>14} {"OK?":>6}')
print('-' * 90)
for col, (lo, hi) in COLUMN_RANGES.items():
    if col in df_clean.columns:
        actual_min = df_clean[col].min()
        actual_max = df_clean[col].max()
        ok = (actual_min >= lo) and (actual_max <= hi)
        status = '✓' if ok else '✗'
        print(f'{col:<35} {actual_min:>8.2f} {actual_max:>8.2f} {lo:>14.2f} {hi:>14.2f} {status:>6}')

## 7. Distribution Shift Check

In [ ]:
check_cols = ['rainfall_mm', 'soil_moisture', 'slope_degree']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, check_cols):
    ax.hist(df_raw[col].dropna(), bins=30, alpha=0.5, label='Raw', color='#e74c3c')
    ax.hist(df_clean[col].dropna(), bins=30, alpha=0.5, label='Cleaned', color='#2ecc71')
    ax.set_title(col, fontweight='bold')
    ax.legend()

plt.suptitle('Distribution — Raw vs Cleaned', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Cleaned Dataset

In [ ]:
processed_dir = REPO_ROOT / 'ml' / 'datasets' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

clean_path = processed_dir / 'landslide_clean.csv'
df_clean.to_csv(clean_path, index=False)
print(f'✓ Cleaned dataset saved to: {clean_path}')
print(f'  Shape: {df_clean.shape}')
print(f'  Next step: Run 03_model_training.ipynb')